In [28]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder , StandardScaler , OneHotEncoder 
import tensorflow as tf
import pickle
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense 
from tensorflow.keras.callbacks import TensorBoard, EarlyStopping

In [5]:
data = pd.read_csv("Churn_Modelling.csv")

data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [7]:
data = data.drop(["RowNumber","CustomerId","Surname"],axis=1)

In [11]:
labelEncoderGenderReg = LabelEncoder()

data["Gender"] = labelEncoderGenderReg.fit_transform(data["Gender"])

data

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,0,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,0,41,1,83807.86,1,0,1,112542.58,0
2,502,France,0,42,8,159660.80,3,1,0,113931.57,1
3,699,France,0,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,0,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...
9995,771,France,1,39,5,0.00,2,1,0,96270.64,0
9996,516,France,1,35,10,57369.61,1,1,1,101699.77,0
9997,709,France,0,36,7,0.00,1,0,1,42085.58,1
9998,772,Germany,1,42,3,75075.31,2,1,0,92888.52,1


In [12]:
oneHotEncoderGeo = OneHotEncoder()

geoEncoded = oneHotEncoderGeo.fit_transform(data[["Geography"]])

geoEncoded

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 10000 stored elements and shape (10000, 3)>

In [13]:
oneHotEncoderGeo.get_feature_names_out(["Geography"])

array(['Geography_France', 'Geography_Germany', 'Geography_Spain'],
      dtype=object)

In [20]:
geoEncodedDf = pd.DataFrame(geoEncoded.toarray(),columns=oneHotEncoderGeo.get_feature_names_out(["Geography"]))

In [21]:
data = pd.concat([data.drop(["Geography"],axis=1),geoEncodedDf],axis=1)

In [24]:
X = data.drop(["EstimatedSalary"],axis=1)
Y = data["EstimatedSalary"]

In [25]:
X_train , X_test , Y_train , Y_test = train_test_split(X,Y,test_size=0.2,random_state=42)

In [26]:
scalerReg = StandardScaler()

X_train = scalerReg.fit_transform(X_train)
X_test = scalerReg.fit_transform(X_test)

In [29]:
with open("labelEncoderGenderReg.pkl","wb") as file:
    pickle.dump(labelEncoderGenderReg,file)

with open("OneHotEncoderGeoReg.pkl","wb") as file:
    pickle.dump(oneHotEncoderGeo,file)

with open("scalerReg.pkl","wb") as file:
    pickle.dump(scalerReg,file)

In [42]:
model = Sequential([
    Dense(64,activation="relu",input_shape=(X_train.shape[1],)),
    Dense(32,activation="relu"),
    Dense(1)
])

In [43]:
model.compile(optimizer="adam",loss="mean_absolute_error",metrics=["mae"])

model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_9 (Dense)                 │ (None, 64)             │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,945 (11.50 KB)

 Trainable params: 2,945 (11.50 KB)

 Non-trainable params: 0 (0.00 B)

In [44]:
import datetime


logsDir = "logs_dir_reg" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")

tensorFlowCallBack = TensorBoard(log_dir=logsDir,histogram_freq=1)

In [45]:
earlyStoppingCallBack = EarlyStopping(monitor="val_loss",patience=10,restore_best_weights=True)

In [46]:
history = model.fit(
    X_train,Y_train,validation_data=(X_test,Y_test),epochs=100,callbacks=[earlyStoppingCallBack,tensorFlowCallBack]
)

Epoch 1/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - loss: 100366.9297 - mae: 100366.9297 - val_loss: 98478.4609 - val_mae: 98478.4609
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 99474.2266 - mae: 99474.2266 - val_loss: 96664.5469 - val_mae: 96664.5469
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 96324.9609 - mae: 96324.9609 - val_loss: 92054.0312 - val_mae: 92054.0312
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 90237.5391 - mae: 90237.5391 - val_loss: 84578.1328 - val_mae: 84578.1328
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 81603.9219 - mae: 81603.9219 - val_loss: 75246.8047 - val_mae: 75246.8047
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 71931.4141 - mae: 71931.4141 - val_loss: 66135.4219 - val_mae: 66135.4219
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 63178.4961 - mae: 63178.4961 - val_loss: 58820.1602 - val_mae: 58820.1602
Epoch 8/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step 

In [48]:
test_loss,test_mae = model.evaluate(X_test,Y_test)
print(f'Test MAE : {test_mae} , Test Loss : ${test_loss}')

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 50237.1836 - mae: 50237.1836
Test MAE : 50237.18359375 , Test Loss : $50237.18359375


In [49]:
model.save("regression_salary.h5")